# ISOM 835 · Explainer — Precision, Recall, ROC and AUC, Plain and Simple
**Suffolk University · Sawyer Business School · Fall 2026 · Prof. Hasan Arslan**

**The big idea:** "How accurate is the model?" is the wrong question for most business problems. Four better questions:

| Question | Name |
|---|---|
| Of the customers we **flagged**, how many really left? | **Precision** |
| Of the customers who **really left**, how many did we flag? | **Recall** |
| How do those two trade off as we move the cutoff? | **ROC curve** |
| Overall, how good is the model at putting risky customers first? | **AUC** |

We start with **10 customers you can check by hand**, then scale up to the real Telco data. Run each cell (Shift + Enter) and read the note above it.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## Part 1 · Ten customers, by hand

A model has scored 10 customers: each gets a **chance of leaving**. We also know what really happened (1 = left, 0 = stayed). The table is sorted from riskiest to safest.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ten = pd.DataFrame({
    'customer': list('ABCDEFGHIJ'),
    'chance':   [0.92, 0.81, 0.74, 0.63, 0.55, 0.42, 0.35, 0.21, 0.12, 0.05],
    'left':     [1,    1,    0,    1,    0,    1,    0,    0,    0,    0],
})
ten

### Step 1. Pick a cutoff, and everyone lands in one of four boxes

Say the cutoff is **0.5**: everyone above 0.5 is **flagged** as "will leave." That's A, B, C, D and E.

| | **Flagged** | **Not flagged** |
|---|---|---|
| **Really left** | ✅ **Caught** (true positive) | ❌ **Missed** (false negative) |
| **Really stayed** | 🔔 **False alarm** (false positive) | 👍 **Correctly left alone** (true negative) |

The next cell sorts the 10 customers into the four boxes.

In [ ]:
cutoff = 0.5
flagged = ten['chance'] >= cutoff
left = ten['left'] == 1

print('✅ caught:      ', list(ten.loc[flagged & left, 'customer']))
print('🔔 false alarm: ', list(ten.loc[flagged & ~left, 'customer']))
print('❌ missed:      ', list(ten.loc[~flagged & left, 'customer']))
print('👍 left alone:  ', list(ten.loc[~flagged & ~left, 'customer']))

### Step 2. Precision: "when we raise the alarm, are we right?"

> **Precision = caught ÷ everyone we flagged**

We flagged 5 customers (A–E), and 3 of them really left (A, B, D). **Precision = 3 / 5 = 60%.**

Think of a **smoke alarm**: precision is how often the alarm means a real fire. Low precision = lots of false alarms = wasted offers, annoyed customers.

### Step 3. Recall: "of everyone who left, how many did we catch?"

> **Recall = caught ÷ everyone who really left**

4 customers really left (A, B, D, F). We caught 3 of them. **Recall = 3 / 4 = 75%.** F slipped through.

Think of a **fishing net**: recall is the share of the fish you actually caught. Low recall = churners walk out the door unnoticed.

The next cell does the same arithmetic in code, so you can check it matches.

In [ ]:
caught = (flagged & left).sum()
precision = caught / flagged.sum()
recall = caught / left.sum()
print(f'precision = {caught} / {flagged.sum()} = {precision:.0%}')
print(f'recall    = {caught} / {left.sum()} = {recall:.0%}')

### Step 4. Move the cutoff, and the two pull against each other

- **Lower cutoff:** flag more people → catch more leavers (**recall ↑**) but more false alarms (**precision ↓**).
- **Higher cutoff:** flag fewer people → fewer false alarms (**precision ↑**) but miss more leavers (**recall ↓**).

Watch it happen on our 10 customers:

In [ ]:
for cutoff in [0.9, 0.7, 0.5, 0.3, 0.1]:
    flagged = ten['chance'] >= cutoff
    caught = (flagged & left).sum()
    precision = caught / flagged.sum()
    recall = caught / left.sum()
    print(f'cutoff {cutoff}: flag {flagged.sum():2d}   precision {precision:4.0%}   recall {recall:4.0%}')

**There's no free lunch.** Which matters more depends on the business:

| Situation | Care most about | Why |
|---|---|---|
| Cheap email offer to stop churn | **Recall** | A false alarm costs a few cents; a lost customer costs $1,500 |
| Freezing a customer's credit card for fraud | **Precision** | A false alarm angers a good customer at the checkout |
| Cancer screening | **Recall** | Missing a case is far worse than a follow-up test |

## Part 2 · The ROC curve: every cutoff in one picture

Instead of picking one cutoff, the **ROC curve** tries **all of them** and plots two numbers for each:

- **Up (y-axis):** recall, the share of **leavers** we caught. Higher is good.
- **Across (x-axis):** the share of **stayers** we falsely flagged (the false-alarm rate). Further left is good.

The next cell draws the ROC curve for our 10 customers. Each dot is one cutoff.

In [ ]:
stayed = ten['left'] == 0
points = [(0.0, 0.0)]
for cutoff in sorted(ten['chance'], reverse=True):
    flagged = ten['chance'] >= cutoff
    false_alarm_rate = (flagged & stayed).sum() / stayed.sum()
    recall = (flagged & left).sum() / left.sum()
    points.append((false_alarm_rate, recall))
xs, ys = zip(*points)

plt.figure(figsize=(4.8, 4.6))
plt.plot(xs, ys, 'o-', color='#2e8b57', lw=2, label='our model')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='coin flip')
plt.xlabel('false-alarm rate (share of stayers flagged)')
plt.ylabel('recall (share of leavers caught)')
plt.title('ROC curve, 10 customers')
plt.legend()
plt.show()

**How to read it:**
- **Bottom-left corner:** cutoff so high that nobody is flagged. No leavers caught, no false alarms.
- **Top-right corner:** cutoff so low that everyone is flagged. All leavers caught, and every stayer falsely flagged too.
- **The dashed diagonal** is a coin flip, a model that knows nothing.
- **A good model hugs the top-left corner:** it catches leavers *before* it starts bothering stayers.

## Part 3 · AUC: one number for the whole curve

**AUC** stands for "**A**rea **U**nder the **C**urve": how much of the square lies under the ROC curve. 1.0 is perfect, and 0.5 is a coin flip.

There's a much friendlier way to say it:

> **AUC = pick one customer who left and one who stayed, at random. How often does the model give the leaver the higher score?**

We have 4 leavers and 6 stayers, so there are 4 × 6 = **24 pairs**. Let's count them:

In [ ]:
leavers = ten.loc[ten['left'] == 1, 'chance']
stayers = ten.loc[ten['left'] == 0, 'chance']

wins = 0
for l in leavers:
    for s in stayers:
        if l > s:
            wins += 1
print(f'leaver ranked higher in {wins} of {len(leavers) * len(stayers)} pairs = {wins / (len(leavers) * len(stayers)):.3f}')

from sklearn.metrics import roc_auc_score
print(f"scikit-learn's roc_auc_score:              {roc_auc_score(ten['left'], ten['chance']):.3f}   ← the same number")

**Same number.** AUC is just "how often the model ranks a leaver above a stayer."

**Two things AUC does NOT tell you:**
1. **Which cutoff to use.** AUC looks at all cutoffs at once; picking one is a business decision (Session 5).
2. **Whether the percentages are honest.** AUC only cares about the *order*. Multiply every score by 0.5 and the AUC doesn't change.

| AUC | Rough meaning |
|---|---|
| 0.5 | Coin flip, useless |
| 0.7 | Useful |
| 0.8 | Good |
| 0.9+ | Excellent (or check for a leak!) |

## Part 4 · Now on the real Telco data

The same ideas on 7,043 real customers. The next cell trains a logistic regression on three facts (tenure, monthly bill, month-to-month contract), **learning on 80%** and **testing on the 20% it has never seen**. Just run it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, RocCurveDisplay

df = pd.read_csv('https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv')
df['left'] = (df['Churn'] == 'Yes').astype(int)
df['month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
X = df[['tenure', 'MonthlyCharges', 'month_to_month']]
X_tr, X_te, y_tr, y_te = train_test_split(X, df['left'], test_size=0.2, stratify=df['left'], random_state=835)

model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
chance = model.predict_proba(X_te)[:, 1]
print(f'{len(y_te):,} test customers, {y_te.sum()} of them left')

**Precision and recall at three cutoffs.** Same pattern as the 10 customers, just bigger numbers:

In [ ]:
for cutoff in [0.7, 0.5, 0.3]:
    flagged = chance >= cutoff
    print(f'cutoff {cutoff}: flag {flagged.sum():4d}   precision {precision_score(y_te, flagged):.0%}   recall {recall_score(y_te, flagged):.0%}')

**The ROC curve and AUC.** scikit-learn draws the curve in one line and prints the AUC in the legend.

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.6))
RocCurveDisplay.from_predictions(y_te, chance, name='logistic regression', ax=ax)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='coin flip')
ax.set_xlabel('false-alarm rate (share of stayers flagged)')
ax.set_ylabel('recall (share of leavers caught)')
ax.legend()
plt.show()
print(f'AUC = {roc_auc_score(y_te, chance):.3f}')
print(f'In words: pick one leaver and one stayer at random, and the model ranks the leaver higher {roc_auc_score(y_te, chance):.0%} of the time.')

> ### Going further (optional): the precision–recall curve
> When the thing you're looking for is **rare** (fraud is often under 1% of transactions), the ROC curve can look great even when most alarms are false. Then plot **precision vs. recall** directly: `PrecisionRecallDisplay.from_predictions(y_te, chance)`. Its baseline isn't the diagonal; it's the share of positives, here about 27%.

## Your turn

1. **By hand.** In Part 1, use a cutoff of **0.3**. Which customers are caught, missed and false alarms? Compute precision and recall on paper, then check with code. *Hint: copy the Step 1 cell and change `cutoff`.*
2. **Pick for the business.** A $5 email offer vs. a $300 phone call from a manager: for each, would you choose a low or a high cutoff on the Telco model, and why? Answer in one sentence each, using the words precision and recall.
3. **Explain AUC to your manager** in one sentence with no jargon. *Hint: use the "pick one leaver and one stayer" idea.*

In [ ]:
# Your turn — work here

## What we learned

- **Precision:** of those we flagged, how many were right? (Smoke alarm: are the alarms real?)
- **Recall:** of those who really left, how many did we catch? (Fishing net: how many fish did we get?)
- **Moving the cutoff trades one for the other.** Which matters more is a business question.
- **The ROC curve** shows every cutoff at once; good models hug the top-left corner.
- **AUC** = how often the model ranks a random leaver above a random stayer. 0.5 is a coin flip, 1.0 is perfect. It doesn't pick the cutoff for you.

| Word | Plain English |
|---|---|
| **True positive** | ✅ Flagged, and really left (caught) |
| **False positive** | 🔔 Flagged, but stayed (false alarm) |
| **False negative** | ❌ Not flagged, but left (missed) |
| **True negative** | 👍 Not flagged, and stayed |
| **Precision** | caught ÷ flagged |
| **Recall** | caught ÷ really left |
| **ROC curve** | Recall vs. false-alarm rate, at every cutoff |
| **AUC** | Chance a random leaver outranks a random stayer |